# Clase 07 — Servir un modelo

**Curso:** MLOps y Monitorización con Python · **Bloque B**
**Tiempo estimado:** 110 minutos · **Capítulo del manual:** `manual/07_servir_modelo.md`

## Qué verás en esta clase

- Reentrenar la `Pipeline` ganadora (regresión logística, `class_weight="balanced"`, `C=1.0`) y guardarla como un artefacto en disco (`modelo.pkl`), tal como la cargaría un servicio real.
- Validar la entrada con un modelo **Pydantic** construido a partir de las columnas de `churn.csv`.
- Construir una API con **FastAPI** con dos endpoints: uno **en línea** (`/predecir`, un cliente) y uno **por lotes** (`/predecir_lote`, una lista de clientes).
- Probarla con `fastapi.testclient.TestClient`, sin abrir ningún puerto ni ejecutar `uvicorn.run` en segundo plano.
- Medir la **latencia real** con `time.perf_counter()`, sobre varias peticiones, con media y mediana.
- Exportar el modelo a **ONNX** con `skl2onnx` y comparar su latencia con la del modelo original, con la misma entrada y el mismo número de llamadas.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook.

## Prerequisitos

- De las clases 4 y 5: la configuración ganadora del modelo de churn (regresión logística, `class_weight="balanced"`, `C=1.0`, F1≈0,3166 sobre el conjunto de prueba). Esta clase la reentrena de cero, no depende de ningún archivo guardado en clases anteriores.
- La carpeta `datos/` del curso, con `churn.csv` dentro, y haber leído `datos/LEEME.md`.
- Python básico: funciones, diccionarios, listas por comprensión.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (incluye FastAPI, skl2onnx y onnxruntime, que quizá no estén preinstalados).

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("skl2onnx", "skl2onnx")
asegurar("onnx", "onnx")
asegurar("onnxruntime", "onnxruntime")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, definimos `cargar_datos`, `SEMILLA = 42` y las columnas de entrada.

In [ ]:
import os
import shutil
import time
import json
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenciamos un aviso de compatibilidad de starlette/httpx (fijo, no aporta nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera

Como en las clases anteriores: creamos `proyecto_churn/` desde cero.

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. De un modelo en un notebook a un modelo servido

Hasta la clase 6, "usar el modelo" significaba llamar a `pipeline.predict(...)` dentro del mismo proceso de Python que lo entrenó. Eso funciona para experimentar, pero un sistema real casi nunca tiene esa suerte: quien necesita una predicción (una aplicación web, otro servicio, un proceso batch nocturno) suele ejecutarse en **otro proceso**, a veces en **otra máquina**, escrito a veces en **otro lenguaje**. La forma estándar de resolverlo es envolver el modelo en una **API**: un programa que recibe peticiones por HTTP, las valida, llama al modelo y devuelve una respuesta, sin que quien llama necesite saber nada de scikit-learn.

> **Nota:** este notebook prueba la API con `fastapi.testclient.TestClient`, que llama a la aplicación ASGI directamente en memoria. No abre ningún puerto ni socket real, y por eso el resultado es determinista y no depende de qué otros procesos estén usando la red de tu ordenador — la forma correcta de probar FastAPI dentro de un notebook o de una suite de tests (a diferencia de ejecutar `uvicorn.run` en segundo plano, que sí abriría un puerto real).

---
## 2. Reentrenar la configuración ganadora y guardarla como artefacto

Reentrenamos exactamente la configuración ganadora de las clases 4 y 5 (regresión logística, `class_weight="balanced"`, `C=1.0`, partición 75/25 con semilla 42).

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# Misma configuración ganadora de las clases 4 y 5: regresión logística, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
f1 = round(f1_score(y_test, y_pred), 4)
auc = round(roc_auc_score(y_test, y_proba), 4)
print("F1 (conjunto de prueba):", f1)
print("AUC (conjunto de prueba):", auc)
assert f1 > 0.25 and auc > 0.6, "el reentrenamiento no reprodujo el modelo esperado"

Guardamos la `Pipeline` entrenada en disco con `joblib.dump`: ese archivo (`modelo.pkl`) es el **artefacto** que una API real cargaría al arrancar, sin volver a entrenar nada. Para comprobar que el artefacto es fiel, lo recargamos en una variable distinta (`MODELO`, con mayúsculas, para distinguirla del `pipeline` recién entrenado) y verificamos que predice exactamente igual.

In [ ]:
# Guardamos la Pipeline entrenada como un archivo, exactamente el artefacto que la API cargará
RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
tamano_kb = round(os.path.getsize(RUTA_MODELO) / 1024, 1)
print("Artefacto guardado en:", RUTA_MODELO, f"(~{tamano_kb} KB)")

# La API nunca importa "pipeline" desde este notebook: carga el archivo, como haría un proceso distinto
MODELO = joblib.load(RUTA_MODELO)
assert np.array_equal(MODELO.predict(X_test), pipeline.predict(X_test))
print("El modelo recargado desde disco predice exactamente igual que el original.")

---
## 3. Validar la entrada con Pydantic

Antes de construir la API definimos un modelo Pydantic, `Cliente`, con exactamente las columnas de `churn.csv` que la Pipeline espera como entrada (todas salvo `abandono`, que es la etiqueta a predecir). Primero comprobamos los valores reales de cada columna categórica, para no inventar categorías que no existen en los datos.

In [ ]:
# Los valores permitidos de cada columna categórica, tomados de los datos reales (no inventados)
print("valores de 'plan':", sorted(df["plan"].unique()))
print("valores de 'contrato':", sorted(df["contrato"].unique()))
print("valores de 'metodo_pago':", sorted(df["metodo_pago"].unique()))

`Literal[...]` en Pydantic rechaza cualquier valor que no esté en la lista, con un error de validación claro, antes de que ese dato llegue siquiera a tocar el modelo. `Field(ge=0)` exige que un número sea mayor o igual que cero (no puede haber una antigüedad negativa). El campo `id_cliente` no participa en la predicción: viaja con la petición solo para poder identificar de qué cliente es cada respuesta.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Cliente(BaseModel):
    """Un modelo Pydantic con exactamente las columnas de entrada de churn.csv (todas salvo 'abandono' e 'id_cliente', que se guarda para identificar la respuesta)."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0, description="meses como cliente, no puede ser negativo")
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    """Convierte un Cliente validado en la fila de una sola columna por variable que la Pipeline espera."""
    datos = cliente.model_dump()
    return pd.DataFrame([datos])[COLUMNAS_ENTRADA]

print("Modelo Pydantic 'Cliente' definido, con las", len(COLUMNAS_ENTRADA), "columnas de entrada.")

### 3.1 Probar la validación con un caso bueno y uno malo

In [ ]:
cliente_ok = {
    "id_cliente": "C_DEMO_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",
    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,
}
c = Cliente(**cliente_ok)
print("Cliente válido:", c)

# Un valor de 'plan' que no existe en los datos: Pydantic lo rechaza ANTES de llegar al modelo
cliente_malo = dict(cliente_ok, plan="diamante")
try:
    Cliente(**cliente_malo)
    raise AssertionError("debía haber lanzado un error de validación")
except Exception as e:
    print("Rechazado como se esperaba. Tipo de error:", type(e).__name__)

**¿Qué observas?** El cliente con `plan="diamante"` nunca llega a convertirse en una fila para el modelo: Pydantic lanza un error de validación (`ValidationError`) en el momento de construir el objeto `Cliente`, antes de cualquier cálculo. Esto es exactamente lo que se busca al validar la entrada: rechazar datos incorrectos lo más cerca posible del origen, con un mensaje claro, en vez de dejar que lleguen al modelo y produzcan un resultado silenciosamente sin sentido.

---
## 4. Construir la API con FastAPI: en línea frente a por lotes

Definimos dos endpoints con el mismo modelo cargado en memoria (`MODELO`): `/predecir`, que recibe UN `Cliente` y responde con UNA predicción (servicio **en línea**), y `/predecir_lote`, que recibe una LISTA de `Cliente` y responde con una LISTA de predicciones (servicio **por lotes**).

In [ ]:
from fastapi import FastAPI
from typing import List

app = FastAPI(title="API de churn (clase 7)")


@app.post("/predecir")
def predecir(cliente: Cliente):
    """Servicio EN LÍNEA: una petición, un cliente, respuesta inmediata."""
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELO.predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


@app.post("/predecir_lote")
def predecir_lote(clientes: List[Cliente]):
    """Servicio POR LOTES (batch): una lista de clientes, una lista de respuestas, sin prisa."""
    filas = pd.DataFrame([c.model_dump() for c in clientes])[COLUMNAS_ENTRADA]
    probabilidades = MODELO.predict_proba(filas)[:, 1]
    return [
        {"id_cliente": c.id_cliente, "probabilidad_abandono": round(float(p), 4), "abandono_previsto": int(p >= 0.5)}
        for c, p in zip(clientes, probabilidades)
    ]


print("API definida, con dos endpoints: /predecir (en línea) y /predecir_lote (por lotes).")

| | En línea (`/predecir`) | Por lotes (`/predecir_lote`) |
|---|---|---|
| Entrada | un `Cliente` | una lista de `Cliente` |
| Cuándo se usa | una aplicación pide una respuesta para UN usuario, AHORA | un proceso nocturno recalcula el riesgo de TODA la cartera de clientes |
| Límite de tiempo | importa mucho: el usuario está esperando | casi no importa: puede tardar minutos |
| Coste por fila | mayor (validación y overhead de la petición por cada fila) | menor (el overhead se reparte entre muchas filas, sección 4.1) |

> **Regla práctica:** si la respuesta la espera una persona en el otro extremo, en el momento, es un servicio en línea y la latencia es una restricción real de producto. Si la respuesta alimenta un proceso que se ejecuta sin nadie esperando (un informe, una tabla que se actualiza cada noche), es un servicio por lotes y lo que importa es el **throughput** (filas por segundo), no la latencia de una fila.

---
## 5. Probar la API con `TestClient`

Probamos primero el caso normal (una fila real del conjunto de prueba) y comprobamos la estructura exacta de la respuesta.

In [ ]:
from fastapi.testclient import TestClient

# TestClient llama a la aplicación ASGI directamente en memoria: NO abre ningún puerto ni socket real,
# por eso es la forma correcta de probar una API FastAPI en un notebook, sin uvicorn.run en segundo plano
cliente_api = TestClient(app)

fila_prueba = X_test.iloc[0].to_dict()
fila_prueba["id_cliente"] = "C_TEST_0001"

respuesta = cliente_api.post("/predecir", json=fila_prueba)
print("código de estado:", respuesta.status_code)
print("cuerpo de la respuesta:", respuesta.json())
assert respuesta.status_code == 200
assert set(respuesta.json().keys()) == {"id_cliente", "probabilidad_abandono", "abandono_previsto"}

### 5.1 Probar los casos de entrada inválida

In [ ]:
fila_invalida = dict(fila_prueba, plan="diamante")
respuesta_mala = cliente_api.post("/predecir", json=fila_invalida)
print("código de estado con plan inválido:", respuesta_mala.status_code)
assert respuesta_mala.status_code == 422, "Pydantic debía rechazar 'diamante' como plan"

fila_negativa = dict(fila_prueba, antiguedad_meses=-3)
respuesta_negativa = cliente_api.post("/predecir", json=fila_negativa)
print("código de estado con antiguedad_meses negativa:", respuesta_negativa.status_code)
assert respuesta_negativa.status_code == 422

**¿Qué observas?** FastAPI traduce automáticamente cualquier error de validación de Pydantic en una respuesta HTTP **422 Unprocessable Entity**, sin que hayamos escrito ninguna línea de código para eso: la validación de la sección 3 y el enrutamiento de FastAPI trabajan juntos. Quien llama a la API recibe un código de error claro, no una excepción interna de scikit-learn ni (peor) una predicción calculada sobre datos corruptos.

---
## 6. Medir la latencia real del servicio en línea

Medimos el tiempo de respuesta de `/predecir` con `time.perf_counter()`, repitiendo la misma petición 200 veces (después de un pequeño "calentamiento"), y reportamos la media y la mediana -- no un solo tiempo aislado, que puede ser una casualidad.

In [ ]:
def medir_latencias_ms(funcion_llamada, n_peticiones):
    """Ejecuta funcion_llamada() n_peticiones veces y devuelve la lista de tiempos en milisegundos."""
    tiempos = []
    for _ in range(n_peticiones):
        inicio = time.perf_counter()
        funcion_llamada()
        tiempos.append((time.perf_counter() - inicio) * 1000)
    return np.array(tiempos)


N_PETICIONES = 200
# Calentamos unas pocas veces antes de medir (la primera llamada siempre incluye costes de inicialización)
for _ in range(10):
    cliente_api.post("/predecir", json=fila_prueba)

latencias_online = medir_latencias_ms(lambda: cliente_api.post("/predecir", json=fila_prueba), N_PETICIONES)
media_online = round(float(latencias_online.mean()), 3)
mediana_online = round(float(np.median(latencias_online)), 3)
print("Latencia del endpoint /predecir sobre", N_PETICIONES, "peticiones -- media:", media_online, "ms, mediana:", mediana_online, "ms")
print("(CPU local, ejemplo ilustrativo: incluye la validación de Pydantic y la capa ASGI de TestClient; los tiempos absolutos varían entre máquinas.)")
assert media_online < 500, "una petición individual no debería tardar medio segundo en esta API tan pequeña"

### 6.1 Por lotes frente a en línea: el overhead se amortiza

In [ ]:
lote_100 = X_test.head(100).to_dict("records")
for i, fila in enumerate(lote_100):
    fila["id_cliente"] = f"C_LOTE_{i:04d}"

inicio = time.perf_counter()
respuesta_lote = cliente_api.post("/predecir_lote", json=lote_100)
tiempo_lote_ms = (time.perf_counter() - inicio) * 1000
print("código de estado del lote:", respuesta_lote.status_code)
assert respuesta_lote.status_code == 200
assert len(respuesta_lote.json()) == 100

tiempo_por_fila_lote = round(tiempo_lote_ms / 100, 4)
print("Lote de 100 clientes en una sola petición:", round(tiempo_lote_ms, 3), "ms en total --", tiempo_por_fila_lote, "ms por fila")
print("Frente a la mediana en línea (una fila por petición):", mediana_online, "ms")

**¿Qué observas?** El tiempo por fila dentro del lote de 100 es mucho menor que la latencia de una petición en línea. La diferencia no es que ONNX ni el modelo sean distintos: es que en una petición en línea, cada fila paga POR SEPARADO el coste fijo de la validación de Pydantic y de la capa HTTP/ASGI, mientras que en el lote ese coste fijo se paga UNA VEZ y se reparte entre las 100 filas. Es la razón concreta por la que un proceso batch nocturno agrupa las filas en lugar de llamar 100 veces a `/predecir`.

---
## 7. Exportar el modelo a ONNX

**ONNX** (Open Neural Network Exchange) es un formato abierto para representar modelos de forma independiente del framework en que se entrenaron. `skl2onnx` convierte una `Pipeline` de scikit-learn (preprocesado + modelo) en un grafo ONNX que se puede ejecutar con `onnxruntime`, un motor de inferencia compilado y optimizado, sin depender de todo el paquete de scikit-learn en el servicio final.

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import StringTensorType, FloatTensorType

# skl2onnx necesita un tipo declarado por cada columna de entrada (aquí van "columna a columna", como espera el ColumnTransformer)
tipos_entrada = [(col, StringTensorType([None, 1])) for col in COLUMNAS_CATEGORICAS]
tipos_entrada += [(col, FloatTensorType([None, 1])) for col in COLUMNAS_NUMERICAS]

modelo_onnx = to_onnx(pipeline, initial_types=tipos_entrada, target_opset=17)
RUTA_ONNX = os.path.join(CARPETA, "modelo.onnx")
with open(RUTA_ONNX, "wb") as f:
    f.write(modelo_onnx.SerializeToString())
tamano_onnx_kb = round(os.path.getsize(RUTA_ONNX) / 1024, 1)
print("Modelo exportado a ONNX en:", RUTA_ONNX, f"(~{tamano_onnx_kb} KB)")

### 7.1 Cargar el modelo ONNX y comprobar que predice lo mismo

In [ ]:
import onnxruntime as ort

sesion_onnx = ort.InferenceSession(RUTA_ONNX, providers=["CPUExecutionProvider"])
print("salidas del modelo ONNX:", [o.name for o in sesion_onnx.get_outputs()])


def fila_a_entradas_onnx(fila):
    """Convierte una fila (dict) en el diccionario de arrays que onnxruntime espera, una entrada por columna."""
    entradas = {col: np.array([[fila[col]]], dtype=object) for col in COLUMNAS_CATEGORICAS}
    entradas.update({col: np.array([[fila[col]]], dtype=np.float32) for col in COLUMNAS_NUMERICAS})
    return entradas


entradas_prueba = fila_a_entradas_onnx(fila_prueba)
salida_onnx = sesion_onnx.run(None, entradas_prueba)
probabilidad_onnx = float(salida_onnx[1][0][1])
probabilidad_sklearn = float(MODELO.predict_proba(cliente_a_fila(Cliente(**fila_prueba)))[0, 1])
print("probabilidad scikit-learn:", round(probabilidad_sklearn, 6))
print("probabilidad ONNX:", round(probabilidad_onnx, 6))
assert abs(probabilidad_sklearn - probabilidad_onnx) < 1e-4, "las dos probabilidades deberían coincidir hasta el redondeo de punto flotante"
print("Mismo resultado (hasta el redondeo de punto flotante): la exportación no cambió lo que el modelo predice.")

---
## 8. Comparar la latencia: scikit-learn frente a ONNX

Comparamos con la MISMA entrada y el MISMO número de llamadas (200), pero llamando directamente a cada modelo -- sin la capa de la API -- para aislar el coste puro de la inferencia del coste de validación y HTTP ya medido en la sección 6.

In [ ]:
# Comparación justa: MISMA entrada (fila_prueba), MISMO número de llamadas, sin la capa de la API en ninguno de los dos casos
fila_prueba_df = cliente_a_fila(Cliente(**fila_prueba))

for _ in range(10):
    pipeline.predict_proba(fila_prueba_df)
    sesion_onnx.run(None, entradas_prueba)

latencias_sklearn = medir_latencias_ms(lambda: pipeline.predict_proba(fila_prueba_df), N_PETICIONES)
latencias_onnx = medir_latencias_ms(lambda: sesion_onnx.run(None, entradas_prueba), N_PETICIONES)

media_sklearn = round(float(latencias_sklearn.mean()), 4)
mediana_sklearn = round(float(np.median(latencias_sklearn)), 4)
media_onnx = round(float(latencias_onnx.mean()), 4)
mediana_onnx = round(float(np.median(latencias_onnx)), 4)

print("scikit-learn (Pipeline.predict_proba),", N_PETICIONES, "llamadas -- media:", media_sklearn, "ms, mediana:", mediana_sklearn, "ms")
print("ONNX Runtime (InferenceSession.run),", N_PETICIONES, "llamadas -- media:", media_onnx, "ms, mediana:", mediana_onnx, "ms")
print("(CPU local, ejemplo ilustrativo: los tiempos absolutos varían entre máquinas y entre ejecuciones; lo que se recalcula aquí es el orden de magnitud, no un milisegundo exacto.)")
assert media_sklearn > 0 and media_onnx > 0

**¿Qué observas?** En esta máquina y para este modelo pequeño (una regresión logística sobre pocas columnas), ONNX Runtime resulta notablemente más rápido que llamar a `predict_proba` sobre la `Pipeline` de scikit-learn: la mayor parte del tiempo que tarda scikit-learn no es el cálculo del modelo en sí, sino el overhead de reconstruir un `DataFrame` de pandas y de pasar por el `ColumnTransformer` en Python puro en cada llamada, algo que el grafo ONNX compilado evita. Esto **no es una ley general**: para modelos mucho más grandes, o con un preprocesado distinto, la diferencia puede ser menor o incluso desaparecer -- por eso este capítulo mide, en vez de asumir, y por eso declaramos explícitamente que la cifra es de una máquina local concreta.

> **Atención:** el resultado honesto de esta sección puede variar de una máquina a otra, y a veces también de una ejecución a otra en la misma máquina. Lo que no debería variar es el orden de las magnitudes ni la explicación de dónde viene la diferencia (overhead de Python/pandas frente a un grafo compilado); si en tu máquina scikit-learn resulta más rápido o similar, sigue siendo un resultado válido -- la conclusión correcta es "mide en tu propio entorno de producción", no "ONNX siempre gana".

---
## 9. Limpieza final de la carpeta de trabajo

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen

- Reentrenamos la configuración ganadora (regresión logística, `class_weight="balanced"`, `C=1.0`) y la guardamos como un **artefacto** (`modelo.pkl`) con `joblib`, tal como lo cargaría un servicio real.
- Un modelo **Pydantic** (`Cliente`) valida la entrada con las columnas reales de `churn.csv`: rechaza categorías inexistentes y valores negativos ANTES de que lleguen al modelo, con un HTTP 422 automático.
- **FastAPI** expone dos endpoints: `/predecir` (servicio **en línea**, un cliente, la latencia importa) y `/predecir_lote` (servicio **por lotes**, una lista, importa el throughput).
- `TestClient` prueba la API en memoria, sin abrir ningún puerto ni ejecutar `uvicorn.run`.
- La latencia real (`time.perf_counter()`, 200 peticiones) del endpoint en línea es del orden de unos pocos milisegundos por petición en esta máquina; el coste por fila cae mucho al agrupar en un lote, porque el overhead fijo se reparte.
- **ONNX** (`skl2onnx` + `onnxruntime`) resultó considerablemente más rápido que scikit-learn puro para este modelo pequeño, con la misma entrada y el mismo número de llamadas -- un resultado medido, no asumido, y que puede no repetirse igual en otra máquina o con otro modelo.

*Continúa con la clase 8: Contenedores y CI/CD.*